# 04 — Train Generator (M5): fine-tune ViT5-base sinh dự thảo mục E-HSMT

Input: prompt ghép [trường trích xuất] + [top-5 chunk truy xuất] + [khung mục cần sinh]
(dùng `GeneratorModule._build_prompt`), target: văn bản mục do Tier 3 (template) sinh
ra làm 'silver reference' ban đầu, sau đó thay bằng bản người dùng đã phê duyệt qua
HITL feedback (`hitl/feedback.py`) khi có đủ dữ liệu thật.
Output: checkpoint tại `models/generator_vit5/`.
Metric: ROUGE-L, BERTScore, so với baseline template filling.

In [ ]:
!git clone https://github.com/dannd/autotender-vn.git /content/autotender-vn 2>/dev/null || echo 'Repo đã tồn tại (chạy lại notebook) hoặc private — kiểm tra quyền truy cập.'
%cd /content/autotender-vn

In [ ]:
!pip install -q transformers datasets rouge-score bert-score accelerate sentencepiece pydantic pydantic-settings pyyaml

In [ ]:
import sys
sys.path.insert(0, '/content/autotender-vn/src')
from autotender.models.generator import SECTION_DEFINITIONS, GeneratorModule
from autotender.schemas import TenderNotice
from autotender.ingest.synth_document import build_synthetic_khlcnt_text
from autotender.models.ner import NERModule

SAMPLES_FILE = '/content/autotender-vn/data/samples/tender_notices.jsonl'
notices = [TenderNotice.model_validate_json(l) for l in open(SAMPLES_FILE, encoding='utf-8')]
print(len(notices), 'notices')

## Sinh cặp (prompt, target) huấn luyện

**LƯU Ý (giới hạn nghiên cứu):** target ở đây là bản Tier 3 (template filling) — dùng để
khởi động (warm-start) mô hình học đúng format/cách trích dẫn, KHÔNG thay thế được việc
gán tay hoặc dùng dữ liệu phản hồi HITL thật để mô hình học phong cách diễn giải tự nhiên
hơn thay vì học thuộc lòng template. Ghi rõ hạn chế này trong báo cáo/MODEL_CARD.md.

In [ ]:
generator = GeneratorModule()
ner = NERModule()
pairs = []
for notice in notices:
    text = build_synthetic_khlcnt_text(notice)
    fields = ner.extract(text)
    for section_id in SECTION_DEFINITIONS:
        citations = generator._retrieve_context(section_id)
        prompt = generator._build_prompt(section_id, fields, citations)
        target = generator.generate_section(section_id, fields).text
        pairs.append({'prompt': prompt, 'target': target})
print(len(pairs), 'training pairs')

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
train_ds = Dataset.from_list(train_pairs)
val_ds = Dataset.from_list(val_pairs)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

MODEL_NAME = 'VietAI/vit5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = tokenizer(batch['prompt'], truncation=True, max_length=768)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(batch['target'], truncation=True, max_length=512)
    inputs['labels'] = labels['input_ids']
    return inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=['prompt', 'target'])
val_tok = val_ds.map(preprocess, batched=True, remove_columns=['prompt', 'target'])
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir='/content/gen_out', num_train_epochs=15, per_device_train_batch_size=4,
    per_device_eval_batch_size=4, eval_strategy='epoch', save_strategy='epoch',
    predict_with_generate=True, logging_steps=5, report_to='none',
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok, data_collator=collator, tokenizer=tokenizer)
trainer.train()
trainer.save_model('/content/models/generator_vit5')
tokenizer.save_pretrained('/content/models/generator_vit5')
print('Checkpoint saved — tải về models/generator_vit5/ trong repo local.')

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score

preds = [trainer.model.generate(**tokenizer(p['prompt'], return_tensors='pt', truncation=True, max_length=768)) for p in val_pairs]
pred_texts = [tokenizer.decode(p[0], skip_special_tokens=True) for p in preds]
refs = [p['target'] for p in val_pairs]

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rouge_l = sum(scorer.score(r, p)['rougeL'].fmeasure for r, p in zip(refs, pred_texts)) / len(refs)
P, R, F1 = bert_score(pred_texts, refs, lang='vi')
print('ROUGE-L:', rouge_l, '| BERTScore F1:', F1.mean().item())

In [ ]:
# Nen checkpoint thanh zip va tai truc tiep ve may (khong can mount Drive)
from google.colab import files
import shutil
shutil.make_archive('generator_vit5', 'zip', '/content/models/generator_vit5')
files.download('generator_vit5.zip')
print('Giai nen generator_vit5.zip vao thu muc models/generator_vit5/ trong repo local.')